In [ ]:
%pip install transformers datasets evaluate accelerate torch huggingface_hub numpy

In [ ]:
from huggingface_hub import interpreter_login

interpreter_login()

# Data Prep

Due to the imbalanced nature of the databse, we want to use a stratified split to enforce the same ratio across train and test as the one in the original dataset distribution. 

In [36]:
from collections import Counter
from datasets import Dataset
from math import isclose
def view_ratio(ds: Dataset) -> dict[str, float]:
    counts = Counter(ds["label"])
    to_string = lambda n:  "safe" if n == 1 else "unsafe"

    return {
        to_string(label): count / len(ds)
        for label, count in counts.items()
    }

def assert_ratios_close(actual: dict[str, float], expected: dict[str, float], abs_tol=1e-3):
    assert actual.keys() == expected.keys()

    for label in expected:
        assert isclose(actual[label], expected[label], abs_tol=abs_tol), (
            f"{label}: expected {expected[label]}, got {actual[label]}"
        )

In [37]:
from datasets import load_dataset, ClassLabel

bash_tool_calls = load_dataset("json", data_files="bash-tool-calls.jsonl", split="train")
bash_tool_calls = bash_tool_calls.cast_column(
    "label",
    ClassLabel(names=["unsafe", "safe"])
)

# The model will be fed this format:
# CWD: /path/to/cwd 
# COMMAND: some-bash
format_input = lambda row: { "text": f"CWD: {row["cwd"]}\nCOMMAND: {row["command"]}" }
bash_tool_calls_formatted = bash_tool_calls.map(format_input, remove_columns=["cwd", "command"])
print(bash_tool_calls_formatted["text"][0])
ratios = view_ratio(bash_tool_calls_formatted)
print(ratios)

# We have ~86% Safe calls and ~14% unsafe calls
# We will do a stratified 80 / 10 / 10 split and keep the above ratio the same across these splits
init_split = bash_tool_calls_formatted.train_test_split(test_size=0.2, stratify_by_column="label", seed=42)

train_ds = init_split["train"]
temp_ds = init_split["test"]

# For test/validation split
test_val_split = temp_ds.train_test_split(test_size=0.5, stratify_by_column="label", seed=42)

val_ds = test_val_split["train"]
test_ds = test_val_split["test"]

# Verify ratios
for _, split_ds in {
    "train": train_ds,
    "test": test_ds,
    "validation": val_ds,
}.items():
    assert_ratios_close(view_ratio(split_ds), ratios)

Generating train split: 0 examples [00:00, ? examples/s]

Casting the dataset:   0%|          | 0/3738 [00:00<?, ? examples/s]

Map:   0%|          | 0/3738 [00:00<?, ? examples/s]

CWD: C:\Users\kainguyen23\.claude\projects
COMMAND: cd "/c/Users/kainguyen23/.claude/projects"
f="/home/kainguyen23/source/repos/DeltaMCP/df3fd314-1191-4de8-8a17-2673648c58e5.jsonl"
echo "=== distinct permissionMode values ==="
grep -o '"permissionMode":"[^"]*"' "$f" | sort | uniq -c
echo ""
echo "=== keys mentioning sandbox / safe / decision / inject across ALL projects ==="
grep -rhoE '"[a-zA-Z_]*([Ss]andbox|[Ss]afe|[Dd]ecision|[Ii]nject|[Aa]pprov|[Dd]anger)[a-zA-Z_]*"' --include=*.jsonl . 2>/dev/null | sort | uniq -c | sort -rn | head -40
{'safe': 0.8611556982343499, 'unsafe': 0.13884430176565007}


# Tokenize



In [38]:

from transformers import AutoTokenizer, DataCollatorWithPadding

base_model = "answerdotai/ModernBERT-base"

tokenizer = AutoTokenizer.from_pretrained(base_model)

tokenize = lambda batch: tokenizer(batch["text"], truncation=True, max_length=512)

train_tok = train_ds.map(tokenize, batched=True, remove_columns=["text"])
test_tok = test_ds.map(tokenize, batched=True, remove_columns=["text"])
val_tok = val_ds.map(tokenize, batched=True, remove_columns=["text"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Map:   0%|          | 0/2990 [00:00<?, ? examples/s]

Map:   0%|          | 0/374 [00:00<?, ? examples/s]

Map:   0%|          | 0/374 [00:00<?, ? examples/s]

# Train

In [39]:
import evaluate
import numpy as np
import torch

f1_metric = evaluate.load("f1")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")
accuracy_metric = evaluate.load("accuracy")

counts = np.bincount(train_ds["label"], minlength=2)
weights = counts.sum() / (2 * counts)
class_weights = torch.tensor(weights, dtype=torch.float)

print("counts:", counts)
print("class_weights:", class_weights)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        "f1": f1_metric.compute(
            predictions=preds,
            references=labels,
            pos_label=0,
        )["f1"],

        "precision": precision_metric.compute(
            predictions=preds,
            references=labels,
            pos_label=0,
        )["precision"],

        "recall": recall_metric.compute(
            predictions=preds,
            references=labels,
            pos_label=0,
        )["recall"],

        "accuracy": accuracy_metric.compute(
            predictions=preds,
            references=labels,
        )["accuracy"],
    }

"""
Applies weighted cross-entropy loss.
We apply a misclassification penalty to account for the imbalanced dataset.
"""
def compute_loss(outputs, labels, num_items_in_batch=None):
    logits = outputs.logits

    return torch.nn.CrossEntropyLoss(
        weight=class_weights.to(logits.device)
    )(logits, labels)

counts: [ 415 2575]
class_weights: tensor([3.6024, 0.5806])


In [40]:
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments

id2label = {
    0: "unsafe",
    1: "safe",
}

label2id = {
    "unsafe": 0,
    "safe": 1,
}

model = AutoModelForSequenceClassification.from_pretrained(
    base_model,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)

# This is running on a Colab T4 GPU and so we optimise for that
training_args = TrainingArguments(
    output_dir= "ModernBERT-bash-classifier",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    num_train_epochs=5,
    weight_decay=0.01,

    fp16=True,
    optim="adamw_torch_fused",

    logging_strategy="steps",
    logging_steps=25,

    eval_strategy="epoch",
    save_strategy="epoch",
    
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    compute_loss_func=compute_loss
)

trainer.train()

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,F1,Precision,Recall,Accuracy
1,0.102987,0.131834,0.944444,0.910714,0.980769,0.983957
2,0.115666,0.082272,0.961538,0.961538,0.961538,0.989305
3,0.110136,0.054255,0.962963,0.928571,1.000000,0.989305
4,0.081239,0.179625,0.970297,1.000000,0.942308,0.991979
5,0.000059,0.086192,0.952381,0.943396,0.961538,0.986631


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=935, training_loss=0.07244008631896187, metrics={'train_runtime': 462.6139, 'train_samples_per_second': 32.316, 'train_steps_per_second': 2.021, 'total_flos': 2647040224985688.0, 'train_loss': 0.07244008631896187, 'epoch': 5.0})

# Eval

In [41]:
trainer.evaluate(test_tok)

Training Loss,Validation Loss,Epoch,F1,Precision,Recall,Accuracy
0.000059,0.119233,5,0.961538,0.961538,0.961538,0.989305


{'eval_loss': 0.11923328042030334,
 'eval_f1': 0.9615384615384616,
 'eval_precision': 0.9615384615384616,
 'eval_recall': 0.9615384615384616,
 'eval_accuracy': 0.9893048128342246}

In [ ]:
trainer.save_model(training_args.output_dir)
tokenizer.save_pretrained(training_args.output_dir)
print("Best checkpoint:", trainer.state.best_model_checkpoint)

trainer.push_to_hub()


# Inference on more real-world examples

In [45]:
from transformers import pipeline
import torch

commands = [
    "CWD: /users/paul/repos/llama\nCOMMAND: cat log.txt",
    "CWD: /home/paul/project\nCOMMAND: rm -rf /tmp/test",
    "CWD: /home/paul/project\nCOMMAND: rm -rf /home/paul/project/logs",
    "CWD: /users/paul/repos/llama\nCOMMAND: rg class",
]

model_dir = training_args.output_dir
device = 0 if torch.cuda.is_available() else -1

classifier = pipeline(
    task="text-classification",
    model=model_dir,
    tokenizer=model_dir,
    device=device,
    truncation=True,
    max_length=512,
)

print("loaded id2label:", classifier.model.config.id2label)
print("loaded label2id:", classifier.model.config.label2id)
print("pipeline:", classifier(commands))


Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

loaded id2label: {0: 'unsafe', 1: 'safe'}
loaded label2id: {'safe': 1, 'unsafe': 0}
pipeline: [{'label': 'safe', 'score': 0.9998307228088379}, {'label': 'unsafe', 'score': 0.9999243021011353}, {'label': 'unsafe', 'score': 0.9975678324699402}, {'label': 'safe', 'score': 0.9997127652168274}]
